In [1]:
epoch_in_minutes = 5

In [31]:
num_jobs = 3
num_machines = 3

# Each job is a list of operations.
# Each operation is a list of (machine_id, duration) pairs.
# jobs_data = [
#     [[(0, 3), (1, 2)], [(0, 2)]],         # Job 0: 2 ops
#     [[(1, 2)], [(0, 4), (1, 3)]]          # Job 1: 2 ops
# ]

jobs_data = [  # task = (machine_id, processing_time)
    [  # Job 0
        [(0, 3), (1, 1), (2, 5)],  # task 0
        [(0, 2), (1, 4), (2, 6)],  # task 1
        [(0, 2), (1, 3), (2, 1)],  # task 2
    ],
    [  # Job 1
        [(0, 2), (1, 3), (2, 4)],
        [(0, 1), (1, 5), (2, 4)],
        [(0, 2), (1, 1), (2, 4)],
    ],
    [  # Job 2
        [(0, 2), (1, 1), (2, 4)],
        [(0, 2), (1, 3), (2, 4)],
        [(0, 3), (1, 1), (2, 5)],
    ],
]
jobs_arrival_epoch = [0, 0, 0]

jobs_arrival_epoch = [0, 1, 2]
CI = [10, 20, 15, 25, 5, 10, 20, 30, 25, 15, 10, 5, 5, 10, 20,
      10, 20, 15, 25, 5, 10, 20, 30, 25, 15, 10, 5, 5, 10, 20,
      10, 20, 15, 25, 5, 10, 20, 30, 25, 15, 10, 5, 5, 10, 20,
      10, 20, 15, 25, 5, 10, 20, 30, 25, 15, 10, 5, 5, 10, 20]  # length must be ≥ horizon
power = [50, 70, 100]  # power consumption of machine 0 and 1

# Compute a rough upper bound on time horizon
horizon = 0
for job_index, job in enumerate(jobs_data):
    for task in job:
        max_task_duration = 0
        for alternative in task:
            max_task_duration = max(max_task_duration, alternative[1] + jobs_arrival_epoch[job_index])
        horizon += max_task_duration

all_tasks = {}
all_machines = [[] for _ in range(num_machines)]

# Makespan minimization

In [32]:
from ortools.sat.python import cp_model

def flexible_jobshop():
    model = cp_model.CpModel()
    # Compute a rough upper bound on time horizon
    horizon = 0
    for job_index, job in enumerate(jobs_data):
        for task in job:
            max_task_duration = 0
            for alternative in task:
                max_task_duration = max(max_task_duration, alternative[1] + jobs_arrival_epoch[job_index])
            horizon += max_task_duration

    all_tasks = {}
    all_machines = [[] for _ in range(num_machines)]
    # Create variables
    for job_id, job in enumerate(jobs_data):
        for task_id, alternatives in enumerate(job):
            for m_id, duration in alternatives:
                suffix = f'_{job_id}_{task_id}_{m_id}'
                start = model.NewIntVar(0, horizon, 'start' + suffix)
                end = model.NewIntVar(0, horizon, 'end' + suffix)
                presence = model.NewBoolVar('presence' + suffix)
                interval = model.NewOptionalIntervalVar(start, duration, end, presence, 'interval' + suffix)

                all_tasks[(job_id, task_id, m_id)] = (start, end, interval, presence)
                all_machines[m_id].append((start, duration, interval))

    # Add constraints
    for job_id, job in enumerate(jobs_data):
        for task_id, alternatives in enumerate(job):
            # Only one machine must be selected for each operation
            presences = [all_tasks[(job_id, task_id, m_id)][3] for m_id, _ in alternatives]
            model.AddExactlyOne(presences)

            # Precedence constraints between operations
            if task_id > 0:
                prev_alts = jobs_data[job_id][task_id - 1]
                curr_alts = alternatives
                for prev_m_id, _ in prev_alts:
                    for curr_m_id, _ in curr_alts:
                        prev_end = all_tasks[(job_id, task_id - 1, prev_m_id)][1]
                        curr_start = all_tasks[(job_id, task_id, curr_m_id)][0]
                        prev_presence = all_tasks[(job_id, task_id - 1, prev_m_id)][3]
                        curr_presence = all_tasks[(job_id, task_id, curr_m_id)][3]
                        model.Add(curr_start >= prev_end).OnlyEnforceIf([prev_presence, curr_presence])
            # first task's start time must be after arrival time
            if task_id == 0:
                curr_alts = alternatives
                for curr_m_id, _ in curr_alts:
                    curr_presence = all_tasks[(job_id, task_id, curr_m_id)][3]
                    curr_start = all_tasks[(job_id, task_id, curr_m_id)][0]
                    model.Add(curr_start >= jobs_arrival_epoch[job_id]).OnlyEnforceIf(curr_presence)
            
            

    # Machine capacity constraints: no overlapping intervals
    for machine_id in range(num_machines):
        machine_tasks = all_machines[machine_id]
        model.AddNoOverlap([interval for _, _, interval in machine_tasks]) # For every pair of intervals in the list, the solver ensures that if both are present, then their execution windows do not overlap

    # Objective: minimize makespan (latest end time)
    all_ends = [end for (_, end, _, _) in all_tasks.values()]
    makespan = model.NewIntVar(0, horizon, 'makespan')
    model.AddMaxEquality(makespan, all_ends)
    model.Minimize(makespan)

    # Solve model
    solver = cp_model.CpSolver()
    status = solver.Solve(model)

    # Output solution
    if status in [cp_model.OPTIMAL, cp_model.FEASIBLE]:
        print(f'Minimized makespan: {solver.Value(makespan)}')
        for (j, t, m), (start, end, interval, presence) in all_tasks.items():
            if solver.Value(presence):
                print(f'Job {j}, Task {t} on Machine {m} → Start: {solver.Value(start)}, End: {solver.Value(end)}')
    else:
        print("No solution found.")

# Run the example
flexible_jobshop()

Minimized makespan: 7
Job 0, Task 0 on Machine 1 → Start: 0, End: 1
Job 0, Task 1 on Machine 0 → Start: 1, End: 3
Job 0, Task 2 on Machine 2 → Start: 5, End: 6
Job 1, Task 0 on Machine 2 → Start: 1, End: 5
Job 1, Task 1 on Machine 0 → Start: 5, End: 6
Job 1, Task 2 on Machine 1 → Start: 6, End: 7
Job 2, Task 0 on Machine 1 → Start: 2, End: 3
Job 2, Task 1 on Machine 0 → Start: 3, End: 5
Job 2, Task 2 on Machine 1 → Start: 5, End: 6


# Carbon consumption minimization

In [30]:
from ortools.sat.python import cp_model

def flexible_jobshop():
    model = cp_model.CpModel()
    # Compute a rough upper bound on time horizon
    horizon = 0
    for job_index, job in enumerate(jobs_data):
        for task in job:
            max_task_duration = 0
            for alternative in task:
                max_task_duration = max(max_task_duration, alternative[1] + jobs_arrival_epoch[job_index])
            horizon += max_task_duration
            
    print(horizon)
    all_tasks = {}
    all_machines = [[] for _ in range(num_machines)]

    # Create variables
    for job_id, job in enumerate(jobs_data):
        for task_id, alternatives in enumerate(job):
            for m_id, duration in alternatives:
                suffix = f'_{job_id}_{task_id}_{m_id}'
                start = model.NewIntVar(0, horizon, 'start' + suffix)
                end = model.NewIntVar(0, horizon, 'end' + suffix)
                presence = model.NewBoolVar('presence' + suffix)
                interval = model.NewOptionalIntervalVar(start, duration, end, presence, 'interval' + suffix)

                all_tasks[(job_id, task_id, m_id)] = (start, end, interval, presence)
                # all_machines[m_id].append((start, duration, interval))
                all_machines[m_id].append((start, duration, interval, presence))

    # Add constraints
    for job_id, job in enumerate(jobs_data):
        for task_id, alternatives in enumerate(job):
            # Only one machine must be selected for each operation
            presences = [all_tasks[(job_id, task_id, m_id)][3] for m_id, _ in alternatives]
            model.AddExactlyOne(presences)

            # Precedence constraints between operations
            if task_id > 0:
                prev_alts = jobs_data[job_id][task_id - 1]
                curr_alts = alternatives
                for prev_m_id, _ in prev_alts:
                    for curr_m_id, _ in curr_alts:
                        prev_end = all_tasks[(job_id, task_id - 1, prev_m_id)][1]
                        curr_start = all_tasks[(job_id, task_id, curr_m_id)][0]
                        prev_presence = all_tasks[(job_id, task_id - 1, prev_m_id)][3]
                        curr_presence = all_tasks[(job_id, task_id, curr_m_id)][3]
                        model.Add(curr_start >= prev_end).OnlyEnforceIf([prev_presence, curr_presence])
            # first task's start time must be after arrival time
            if task_id == 0:
                curr_alts = alternatives
                for curr_m_id, _ in curr_alts:
                    curr_presence = all_tasks[(job_id, task_id, curr_m_id)][3]
                    curr_start = all_tasks[(job_id, task_id, curr_m_id)][0]
                    model.Add(curr_start >= jobs_arrival_epoch[job_id]).OnlyEnforceIf(curr_presence)
            
            

    # Machine capacity constraints: no overlapping intervals
    for machine_id in range(num_machines):
        machine_tasks = all_machines[machine_id]
        model.AddNoOverlap([interval for _, _, interval, _ in machine_tasks]) # For every pair of intervals in the list, the solver ensures that if both are present, then their execution windows do not overlap

    # Carbon-aware objective
    total_carbon_terms = []
    active_vars = []
    b_vars = []
    for t in range(horizon):
        active_vars.append([])
        for m in range(num_machines):
            active = model.NewBoolVar(f'active_{t}_{m}')
            active_vars[t].append(active)
            relevant_presences = []
            for (start, dur, interval, presence) in all_machines[m]:
                b = model.NewBoolVar(f'running_{t}_{m}') # b[t][m] = True when a task is running on machine m and the time t is within the [start, end] of the task
                # model.Add(start <= t).OnlyEnforceIf(b)
                # model.Add(t < start + dur).OnlyEnforceIf(b)
                in_window1 = model.NewBoolVar('')
                in_window2 = model.NewBoolVar('')
                model.Add(start <= t).OnlyEnforceIf(in_window1)
                model.Add(start > t).OnlyEnforceIf(in_window1.Not())
                model.Add(t < start + dur).OnlyEnforceIf(in_window2)
                model.Add(t >= start + dur).OnlyEnforceIf(in_window2.Not())
                # model.AddImplication(presence.Not(), b.Not()) # if the task is not running on machine m, then b is False
                model.AddImplication(b, in_window1)
                model.AddImplication(b, in_window2)
                model.AddImplication(b, presence)
                model.AddBoolOr(in_window1.Not(), in_window2.Not(), presence.Not(), b)
                relevant_presences.append(b)
            b_vars.append(relevant_presences)

            # model.AddBoolOr(relevant_presences).OnlyEnforceIf(active) # active[t][m] = True iff at least one task is active at time t on machine m
            # model.AddBoolAnd([b.Not() for b in relevant_presences]).OnlyEnforceIf(active.Not())
            
            model.AddMaxEquality(active, relevant_presences)

            # Compute carbon emission at this time step
            carbon_term = model.NewIntVar(0, CI[t] * power[m], f'carbon_{t}_{m}')
            model.AddMultiplicationEquality(carbon_term, [active, CI[t] * power[m]])
            total_carbon_terms.append(carbon_term)

    total_carbon = model.NewIntVar(0, 1000000, 'total_carbon')
    model.Add(total_carbon == sum(total_carbon_terms))
    model.Minimize(total_carbon)

    # Solve model
    solver = cp_model.CpSolver()
    status = solver.Solve(model)

    # Output solution
    if status in [cp_model.OPTIMAL, cp_model.FEASIBLE]:
        print(f"✅ Total carbon emitted: {solver.Value(total_carbon)}")
        for (j, t, m), (start, end, _, presence) in all_tasks.items():
            if solver.Value(presence):
                print(f'Job {j}, Task {t} on Machine {m} → Start: {solver.Value(start)}, End: {solver.Value(end)}')
    else:
        print("❌ No solution found.")
    
    for t in range(horizon):
        for m in range(num_machines):
            # print(f"Machine {m} at time {t} b: {solver.Value(b_vars[t][m])}")
            print(f"Machine {m} at time {t} Active: {solver.Value(active_vars[t][m])}")
    print("##################")
    for t in range(horizon):
        for m in range(num_machines):
            print(f"Machine {m} at time {t} b: {solver.Value(b_vars[t][m])}")
            # print(f"Machine {m} at time {t} Active: {solver.Value(b_vars[t][m])}")

# Run the example
flexible_jobshop()

49
✅ Total carbon emitted: 3900
Job 0, Task 0 on Machine 1 → Start: 4, End: 5
Job 0, Task 1 on Machine 0 → Start: 11, End: 13
Job 0, Task 2 on Machine 2 → Start: 19, End: 20
Job 1, Task 0 on Machine 0 → Start: 4, End: 6
Job 1, Task 1 on Machine 0 → Start: 19, End: 20
Job 1, Task 2 on Machine 1 → Start: 26, End: 27
Job 2, Task 0 on Machine 1 → Start: 11, End: 12
Job 2, Task 1 on Machine 0 → Start: 26, End: 28
Job 2, Task 2 on Machine 1 → Start: 34, End: 35
Machine 0 at time 0 Active: 0
Machine 1 at time 0 Active: 0
Machine 2 at time 0 Active: 0
Machine 0 at time 1 Active: 0
Machine 1 at time 1 Active: 0
Machine 2 at time 1 Active: 0
Machine 0 at time 2 Active: 0
Machine 1 at time 2 Active: 0
Machine 2 at time 2 Active: 0
Machine 0 at time 3 Active: 0
Machine 1 at time 3 Active: 0
Machine 2 at time 3 Active: 0
Machine 0 at time 4 Active: 1
Machine 1 at time 4 Active: 1
Machine 2 at time 4 Active: 0
Machine 0 at time 5 Active: 1
Machine 1 at time 5 Active: 0
Machine 2 at time 5 Active: 0
